# Merged vs Non-Merged — analiza offline

Notebook działa **bez uruchomionych serwisów** — importuje funkcje z `backend/app.py`
i modele z `similarity_backend/app.py` bezpośrednio w procesie.

**Struktura plików:**
```
./                                 ← notebook + analyze_merged_vs_nonmerged.py
./backend/app.py
./similarity_backend/app.py
```

**Pierwsze uruchomienie** — ładowanie modeli BGE-M3 + BGE-reranker trwa ~30s.
Kolejne komórki korzystają z cache, modele nie są ładowane ponownie.


## 1. Parametry

In [2]:
# ── ZMIEŃ TU ──────────────────────────────────────────────────────────────────

DATASET            = "gutenqa_all"   # slug datasetu (merged lub non-merged)
DATA_ROOT          = "./data"       # ścieżka do katalogu z pirb/ i processed/

SCORE_TOL          = 0.02            # tolerancja retrieval score ±
RERANKER_THRESHOLD = 0.5             # minimalny score rerankera dla merged chunków

VERBOSE            = False           # True = drukuj każdy case na bieżąco


## 2. Import modułu analizy

In [3]:
import importlib.util, sys, json
from pathlib import Path
from dataclasses import asdict

# Remove cached module so re-running this cell picks up any changes to the .py file
sys.modules.pop("analysis", None)

spec = importlib.util.spec_from_file_location("analysis", Path("analyze_merged_vs_nonmerged.py"))
mod  = importlib.util.module_from_spec(spec)
sys.modules["analysis"] = mod   # register before exec (same fix as inside the script)
spec.loader.exec_module(mod)
analysis = mod

merged_slug, nonmerged_slug = analysis._get_pair_slugs(DATASET)
print(f"Merged dataset    : {merged_slug}")
print(f"Non-merged dataset: {nonmerged_slug}")
print(f"Data root         : {Path(DATA_ROOT).resolve()}")


Merged dataset    : gutenqa_all_merged
Non-merged dataset: gutenqa_all
Data root         : C:\Users\admin\Downloads\chunker-analyzer\chunker-analyzer-v8\data


## 3. Sprawdź dostępne datasety (opcjonalnie)

In [4]:
b = analysis._load_backend(DATA_ROOT)
groups = b.group_experiments_by_dataset()
print(f"Dostępne datasety ({len(groups)}):")
for slug, exps in sorted(groups.items()):
    chunkers = [b.get_experiment_metadata(e).get("chunker_name", e) for e in exps]
    print(f"  {slug}  [{', '.join(chunkers)}]")


Dostępne datasety (11):
  gutenqa_all  [fixed_size, max_min_chunker, sequential_hac_chunker, recursive_semantic, graphseg]
  gutenqa_all_merged  [fixed_size, max_min_chunker, recursive_semantic, sequential_hac_chunker]
  literaryqa_test  [fixed_size, max_min_chunker, sequential_hac_chunker, recursive_semantic]
  natural_questions_validation_300_docs  [fixed_size, max_min_chunker, sequential_hac_chunker, text_tiling, recursive_semantic, graphseg, lumberchunker_api, dense_x_retrieval]
  novelqa_public  [fixed_size, max_min_chunker, recursive_semantic, sequential_hac_chunker]
  poquad_validation  [fixed_size, max_min_chunker, sequential_hac_chunker, text_tiling, recursive_semantic, graphseg, lumberchunker_api, dense_x_retrieval]
  poquad_validation_merged  [fixed_size, max_min_chunker, recursive_semantic, sequential_hac_chunker]
  qasper_test  [fixed_size, max_min_chunker, sequential_hac_chunker, text_tiling, graphseg, lumberchunker, recursive_semantic, lumberchunker_api, dense_x_retrieva

## 4. Załaduj modele ML (jednorazowo ~30s)

In [5]:
# Jawne załadowanie — żeby nie zaskoczyło podczas analizy
sim = analysis._load_similarity()
print(f"Device: {sim.device}")
print(f"Embed model : {sim.embed_model}")
print(f"Reranker    : {sim.reranker}")


Loading ML models (BGE-M3 + BGE-reranker) — first call only, ~30s…


c:\Users\admin\Downloads\chunker-analyzer\chunker-analyzer-v8\similarity_backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
  Models loaded on device: cuda
Device: cuda
Embed model : SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
Reranker    : <FlagEmbedding.inference.reranker.encoder_only.base.BaseReranker object at 0x000001943AE3C2C0>


## 5. Uruchom analizę

In [6]:
cases = analysis.analyze(
    dataset_slug       = DATASET,
    data_root          = DATA_ROOT,
    score_tol          = SCORE_TOL,
    reranker_threshold = RERANKER_THRESHOLD,
    verbose            = VERBOSE,
)
print(f"\nZnaleziono łącznie: {len(cases)} pasujących przypadków.")


Merged dataset    : gutenqa_all_merged
Non-merged dataset: gutenqa_all
Data root         : C:\Users\admin\Downloads\chunker-analyzer\chunker-analyzer-v8\data

Loading queries for merged dataset…
  2981 queries
Loading queries for non-merged dataset…
  2981 queries
Loading metrics…
  Chunkers: ['fixed_size', 'max_min_chunker', 'recursive_semantic', 'sequential_hac_chunker']


── Chunker: fixed_size ─────────────────────────────


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 330.31it/s]
c:\Users\admin\Downloads\chunker-analyzer\chunker-analyzer-v8\similarity_backend\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2888: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 503.46it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 999.83it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 1000.79it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 882.45it/s]


...

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 490.91it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 500.04it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 502.31it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 420.57it/s]



  Found 5 matching cases for sequential_hac_chunker


Znaleziono łącznie: 88 pasujących przypadków.


## 6. Podsumowanie tekstowe

In [7]:
analysis.print_summary(cases, SCORE_TOL, RERANKER_THRESHOLD)



SUMMARY  —  88 matching case(s)
  |retrieval_diff| ≤ 0.02  AND  |reranker_diff| > 0.5

Chunker: fixed_size  (33 case(s))

  Query q.1000: What natural phenomenon is described as having a certain beauty when it is in a state that…
  Non-merged HIT chunks:
    [55164]  ret=0.5557  rer=-2.8672
      irst made all even and uniform, they become it well nevertheless, and have a certain peculiar property, to stir the appe…

  Merged conflicting chunks (embedding≈, reranker≠):
    [50109]  ret=0.5713  rer=-4.9453
      roth of a foaming wild boar, and many other like things, though by themselves considered, they are far from any beauty, …

  Query q.1039: What physical discomfort was Mr. Casaubon experiencing in the library?
  Non-merged HIT chunks:
    [57152]  ret=0.5259  rer=3.4805
      rity of the outdoor snow. As she laid the cameo-cases on the table in the bow-window, she unconsciously kept her hands o…

  Merged conflicting chunks (embedding≈, reranker≠):
    [51939]  ret=0.5205  rer=

## 7. Wyniki jako DataFrame

In [9]:
import pandas as pd

rows = []
for c in cases:
    for mc in c.merged_close_chunks:
        rows.append({
            "query_id":            c.query_id,
            "query_text":          c.query_text[:80] + "…" if len(c.query_text) > 80 else c.query_text,
            "chunker":             c.chunker_name,
            "nm_best_ret":         round(c.nonmerged_best_retrieval, 4),
            "merged_chunk_id":     mc.chunk_id,
            "merged_ret":          round(mc.score_retrieval, 4),
            "merged_rer":          round(mc.score_reranker,  4),
            "ret_diff":            round(abs(mc.score_retrieval - c.nonmerged_best_retrieval), 4),
            "merged_chunk_preview": mc.contents_preview[:80],
        })

df = pd.DataFrame(rows)
if df.empty:
    print("Brak wyników — spróbuj zwiększyć SCORE_TOL lub zmniejszyć RERANKER_THRESHOLD")
else:
    print(f"Wierszy: {len(df)},  unikalnych pytań: {df['query_id'].nunique()}")
    display(df)


Wierszy: 128,  unikalnych pytań: 25


,query_id,query_text,chunker,nm_best_ret,merged_chunk_id,merged_ret,merged_rer,ret_diff,merged_chunk_preview
0,q.triviaqa-span-annotated.165845,What was the word for an Indian soldier servin...,fixed_size,0.4688,5306,0.4580,-6.0977,0.0107,"rust remains. August 2, 1858 Parliament puts I..."
1,q.triviaqa-span-annotated.263842,Which British army regiment are known as Sappers,fixed_size,0.3855,12535,0.3887,-6.8359,0.0032,am Tuesday we had arrived at Camp Bastion to s...
2,q.triviaqa-span-annotated.288148,"Batting, Cornerstones, Sashing and Layer Cake ...",fixed_size,0.4146,14039,0.3972,-4.9531,0.0173,few large pieces of fabric pieced to slightly ...
3,q.triviaqa-span-annotated.298469,"What is the next in this sequence: Madison, Mo...",fixed_size,0.4187,4417,0.3989,-5.7383,0.0198,rural coal-mining towns in Eastern Kentucky. S...
4,q.triviaqa-span-annotated.300459,What type of hat was famously worn by Che Guev...,fixed_size,0.4546,5864,0.4727,-4.4453,0.0181,"GREEN BERET is also called ""green beanie"" and ..."
...,...,...,...,...,...,...,...,...,...
123,q.triviaqa-span-annotated.80584,In the 2001 official UK national census what r...,dense_x_retrieval,0.5073,127342,0.5073,-9.5156,0.0000,
124,q.triviaqa-span-annotated.80584,In the 2001 official UK national census what r...,dense_x_retrieval,0.5073,127347,0.5073,-9.5156,0.0000,
125,q.triviaqa-span-annotated.80584,In the 2001 official UK national census what r...,dense_x_retrieval,0.5073,127358,0.5073,-9.5156,0.0000,
126,q.triviaqa-span-annotated.80584,In the 2001 official UK national census what r...,dense_x_retrieval,0.5073,127577,0.5073,-9.5156,0.0000,


## 8. Sortowanie i filtrowanie

In [10]:
if not df.empty:
    # Najwyższy reranker score merged na górze
    display(df.sort_values("merged_rer", ascending=False).reset_index(drop=True))


,query_id,query_text,chunker,nm_best_ret,merged_chunk_id,merged_ret,merged_rer,ret_diff,merged_chunk_preview
0,q.triviaqa-span-annotated.300459,What type of hat was famously worn by Che Guev...,fixed_size,0.4546,5864,0.4727,-4.4453,0.0181,"GREEN BERET is also called ""green beanie"" and ..."
1,q.triviaqa-span-annotated.36766,"January King, Meteor, and Salarite are varieti...",fixed_size,0.3694,25352,0.3562,-4.5352,0.0132,abbage Cool-Season Vegetables: How to Grow Cab...
2,q.triviaqa-span-annotated.288148,"Batting, Cornerstones, Sashing and Layer Cake ...",fixed_size,0.4146,14039,0.3972,-4.9531,0.0173,few large pieces of fabric pieced to slightly ...
3,q.triviaqa-span-annotated.57518,In which Bond film does Charles Grey play the ...,fixed_size,0.4360,25990,0.4487,-5.1602,0.0127,D IS BACK - Sean Connery is BOND [British adva...
4,q.triviaqa-span-annotated.292764,Suint is a natural grease formed from the drie...,max_min_chunker,0.4248,20980,0.4204,-5.6836,0.0044,The breed's double coat enables it to adapt to...
...,...,...,...,...,...,...,...,...,...
123,q.triviaqa-span-annotated.335736,"Who famously ate peanut butter, banana, and ba...",dense_x_retrieval,0.4583,149980,0.4412,-10.0156,0.0171,
124,q.triviaqa-span-annotated.335736,"Who famously ate peanut butter, banana, and ba...",dense_x_retrieval,0.4583,149960,0.4412,-10.0156,0.0171,
125,q.triviaqa-span-annotated.335736,"Who famously ate peanut butter, banana, and ba...",dense_x_retrieval,0.4583,149961,0.4412,-10.0156,0.0171,
126,q.triviaqa-span-annotated.270507,Which popular garden bird has the Latin name C...,dense_x_retrieval,0.4946,-1,0.4873,-10.0469,0.0073,


In [11]:
# Filtruj: chunki gdzie reranker merged > 3.0
if not df.empty:
    high = df[df["merged_rer"] > 3.0].sort_values("merged_rer", ascending=False)
    print(f"Chunki z reranker merged > 3.0: {len(high)}")
    display(high.reset_index(drop=True))


Chunki z reranker merged > 3.0: 0


,query_id,query_text,chunker,nm_best_ret,merged_chunk_id,merged_ret,merged_rer,ret_diff,merged_chunk_preview


## 9. Wykres — liczba przypadków per chunker

In [13]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp312-cp312-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pillow-12.2.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.2 MB 10.5 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.2 MB 12.0 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.2 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 12.4 MB/s eta 0:00:00
Using cached contourpy-1.3.3-cp312-cp312-win

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [15]:
import matplotlib.pyplot as plt

if not df.empty:
    counts = df.groupby("chunker")["query_id"].nunique().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(max(6, len(counts)*1.8), 4))
    counts.plot(kind="bar", ax=ax, color="#5c7cfa", edgecolor="#3a5de0")
    ax.set_title(
        f"Pytania: non-merged lepszy, merged miał bliski chunk (±{SCORE_TOL}) "
        f"z wysokim reranker (>{RERANKER_THRESHOLD})"
    )
    ax.set_xlabel("Chunker")
    ax.set_ylabel("Liczba unikalnych pytań")
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(counts):
        ax.text(i, v + 0.05, str(v), ha="center")
    plt.tight_layout()
    plt.show()


ModuleNotFoundError: No module named 'matplotlib'

## 10. Scatter — retrieval vs reranker score (merged chunki)

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    chunkers = df["chunker"].unique()
    colors   = ["#5c7cfa", "#f59f00", "#63e6be", "#f03e3e", "#da77f2"]

    fig, ax = plt.subplots(figsize=(7, 5))
    for i, ch in enumerate(chunkers):
        sub = df[df["chunker"] == ch]
        ax.scatter(sub["merged_ret"], sub["merged_rer"],
                   label=ch, color=colors[i % len(colors)],
                   alpha=0.75, s=60, edgecolors="white", linewidths=0.5)

    ax.axhline(RERANKER_THRESHOLD, color="red",  linestyle="--", lw=1.2,
               label=f"reranker threshold ({RERANKER_THRESHOLD})")
    ax.axvspan(
        df["nm_best_ret"].min() - SCORE_TOL,
        df["nm_best_ret"].max() + SCORE_TOL,
        alpha=0.06, color="steelblue", label=f"retrieval ±{SCORE_TOL} band",
    )
    ax.set_xlabel("Score Retrieval (merged chunk)")
    ax.set_ylabel("Score Reranker (merged chunk)")
    ax.set_title("Merged chunki spełniające oba warunki")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## 11. Podgląd pojedynczego przypadku

In [ ]:
CASE_IDX = 0   # zmień żeby zobaczyć inny przypadek

if cases:
    c = cases[CASE_IDX]
    print(f"Query ID  : {c.query_id}")
    print(f"Chunker   : {c.chunker_name}")
    print(f"Pytanie   : {c.query_text}")
    print()
    print("Non-merged — zwrócone relevantne chunki:")
    for ch in c.nonmerged_hit_chunks:
        print(f"  [{ch.chunk_id}]  ret={ch.score_retrieval:.4f}  rer={ch.score_reranker:.4f}")
        print(f"    {ch.contents_preview}")
    print()
    print(f"Merged — spełniające (ret±{SCORE_TOL}, rer>{RERANKER_THRESHOLD}):")
    for ch in c.merged_close_chunks:
        diff = abs(ch.score_retrieval - c.nonmerged_best_retrieval)
        print(f"  [{ch.chunk_id}]  ret={ch.score_retrieval:.4f} (diff={diff:.4f})  rer={ch.score_reranker:.4f}")
        print(f"    {ch.contents_preview}")
else:
    print("Brak przypadków.")


## 12. Szybka re-analiza z innymi parametrami

Modele są już załadowane — ponowna analiza nie wymaga ładowania modeli.

In [ ]:
# Zmień progi i uruchom ponownie bez przeładowania modeli
cases2 = analysis.analyze(
    dataset_slug       = DATASET,
    data_root          = DATA_ROOT,
    score_tol          = 0.05,         # ← zmień
    reranker_threshold = 0.5,          # ← zmień
    verbose            = False,
)
print(f"Nowe parametry → {len(cases2)} przypadków")
analysis.print_summary(cases2, 0.05, 0.5)


## 13. Zapis wyników do JSON

In [ ]:
OUTPUT_FILE = f"results_{DATASET}_tol{SCORE_TOL}_rer{RERANKER_THRESHOLD}.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump([asdict(c) for c in cases], f, indent=2, ensure_ascii=False)
print(f"Zapisano {len(cases)} przypadków → {OUTPUT_FILE}")
